In [1]:
#Higher Level Implementation
import torch
import torch.nn as nn

class custom_module_higher(nn.Module):
    def __init__(self,input_size , output_size):
        super().__init__()
        self.Linear = nn.Linear(input_size , output_size)
        self.ReLu = nn.ReLU()

    def forward(self , x):
        x = self.Linear(x)
        x = self.ReLu(x)
        return x
    

#Lower Level Implementation

class custom_module_Lowe(nn.Module):
    def __init__(self , input_size , ouput_size):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(ouput_size , input_size))
        self.bias = nn.Parameter(torch.randn(ouput_size))
    
    def forward(self ,x):
        x = x  @ self.weight.T + self.bias
        x = torch.clamp(x , min=0)
        return x

In [2]:
from sklearn.datasets import fetch_covtype
from torch.utils.data import TensorDataset

covtype = fetch_covtype()
x_covtype = torch.tensor(covtype.data , dtype = torch.float32)
means = x_covtype.mean(dim=0 , keepdim=True)
std = x_covtype.std(dim=0 , keepdim=True)
x_covtype = (x_covtype - means) / std #Standariztion 

y_covtype = torch.tensor(covtype.target , dtype=torch.float32)

covtype_dataset = TensorDataset(x_covtype , y_covtype)


In [3]:
#Train_dataset,Test_datset,Valid_datset
from torch.utils.data import random_split

train_len = len(covtype_dataset) * 80 // 100 #// Integer Divison Converts float into int
valid_len = len(covtype_dataset) * 10 // 100 
test_len = len(covtype_dataset) - train_len - valid_len

train_dataset , valid_datset , test_dataser = random_split(covtype_dataset , [train_len , valid_len , test_len])

In [7]:
#Data_model
n_inputs = len(covtype.feature_names)
n_outputs = len(set(covtype.target))

model = nn.Sequential(
    nn.Linear(n_inputs , 200),
    nn.ReLU(),
    nn.Linear(200,100),
    nn.ReLU(),
    nn.Linear(100 , n_outputs)
).to(device="cuda")

In [ ]:
#Training Model
def train(model , optimizer , criterion , data_loader , n_epochs):
    model.train()
    for epoch in n_epochs:
        total_loss = 0.0
        for x_batch , y_batch in data_loader:
            x_batch , y_batch = x_batch.to(device = "cuda") , y_batch.to(device = "cuda")
            y_pred = model(x_batch)
            loss = criterion(y_pred , y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(data_loader)
        print(f"Epochs : {epoch + 1} / {n_epochs} : Loss {mean_loss}")

#Testing Model
def evaluate(model , data_loader , metrics_fn ,aggregate_fn = lambda metric : torch.mean(metric)):
    model.eval()
    metrics = []
    for x_batch , y_batch in data_loader:
        x_batch , y_batch = x_batch.to(device = "cuda") , y_batch.to(device = "cuda")
        y_pred = model(x_batch)
        metric = metrics_fn(y_pred , y_batch)
        metrics.append(metric)
    return aggregate_fn(torch.stack(metrics))

